# 01 — Train Stage 1

Train only `A1_IFCRN_PP`. Five outer folds create leakage-safe OOF predictions. For each outer fold, one separate inner fold is used for early stopping and threshold selection.


In [ ]:
#from google.colab import drive
#drive.mount('/content/drive')

from pathlib import Path
import sys

#PROJECT_DIR = Path('/content/drive/MyDrive/Research/PUMA')
PROJECT_DIR = Path.cwd().resolve()

%cd {PROJECT_DIR}
sys.path.insert(0, str(PROJECT_DIR))


In [ ]:
%pip install -q -r requirements_colab.txt


In [ ]:
from puma.runtime import create_runtime, preflight_environment

runtime = create_runtime(
    PROJECT_DIR,
    run_folds=(0, 1, 2, 3, 4),  # required for complete Stage-1 OOF coverage
    epochs=30,
    effective_batch_size=32,
    stage1_micro_batch_size=16,
    preprocessing_workers=0,
    early_stopping_enabled=True,
    early_stopping_patience=10,
    early_stopping_min_delta=0.0,
)

print(runtime.as_dict())
preflight_report = preflight_environment(
    runtime, require_dataset=True, require_training_dependencies=True
)


In [ ]:
from puma.training.stage1 import run_stage1_a1
from puma.pipeline.final_v13 import write_fixed_stage1_lock_v13

stage1_run_summary = run_stage1_a1(runtime)
stage1_lock = write_fixed_stage1_lock_v13(runtime)

print('Stage-1 model:', stage1_lock['selected_experiment'])
print('OOF folds:', stage1_lock['run_folds'])
stage1_run_summary
